In [1]:
import ee
import time
import geemap
import os

In [4]:
ee.Authenticate()


Successfully saved authorization token.


In [5]:
# ee.Authenticate() # Uncomment if needed
ee.Initialize(project='e4e-mangrove')

print("Initializing Filtered Direct-to-Disk Pipeline...")

Initializing Filtered Direct-to-Disk Pipeline...


In [6]:
# ==========================================
# 1. Base Geometry
# ==========================================
us_states = ee.FeatureCollection("TIGER/2018/States")
florida = us_states.filter(ee.Filter.eq('NAME', 'Florida'))
florida_bounds = florida.geometry().bounds()

In [7]:
# ==========================================
# 2. Define the Target Label (ESA WorldCover)
# ==========================================
esa = ee.ImageCollection('ESA/WorldCover/v200').first()

# A. The FULL 11-Class Label (This becomes Band 69)
# Values include: 10(Trees), 50(Built-up), 80(Water), 95(Mangroves), etc.
esa_all_classes = esa.select('Map').byte().rename('ESA_MultiClass_Label')

# B. The STRICT Mangrove Filter (Class 95)
# We use this purely to find the coastal bounding boxes
esa_mangrove_filter = esa.eq(95)

In [8]:
# ==========================================
# 3. FILTER THE GRID (The Gatekeeper)
# ==========================================
base_grid = florida_bounds.coveringGrid('EPSG:3857', 20480)

# Scan the grid using the STRICT Class 95 filter to drop the inland noise
tiles_with_stats = esa_mangrove_filter.reduceRegions(
    collection=base_grid,
    reducer=ee.Reducer.max(),
    scale=100,
    tileScale=4 
)
# ONLY keep tiles that have true ESA Mangroves
mangrove_tiles = tiles_with_stats.filter(ee.Filter.gt('max', 0))

In [9]:
# ==========================================
# 4. Build the 69-Band Training Stack
# ==========================================
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(florida_bounds)
      .filterDate('2023-01-01', '2024-01-01')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
      .median())

s2_optical = s2.select(['B4', 'B3', 'B2', 'B8']).rename(['Red', 'Green', 'Blue', 'NIR'])

embeddings = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
              .filterBounds(florida_bounds)
              .filterDate('2023-01-01', '2024-01-01')
              .mosaic())

# Stack: Optical (1-4) + Embeddings (5-68) + FULL Multi-Class Labels (69)
training_stack = s2_optical.addBands(embeddings).addBands(esa_all_classes)

In [11]:
# ==========================================
# 5. Direct-to-Disk Download Loop
# ==========================================
print("Calculating tile intersections via Raster Engine...")
tile_list = mangrove_tiles.toList(500)

def get_coords_gps(tile):
    return ee.Feature(tile).geometry().transform('EPSG:4326', 1).coordinates()

tile_coordinates = tile_list.map(get_coords_gps)
coords_array = tile_coordinates.getInfo()

total_tiles = len(coords_array)
print(f"\nSUCCESS! Total Intersecting Tiles Found: {total_tiles}")

out_dir = os.path.join(os.getcwd(), 'Florida_Training_Dataset')
os.makedirs(out_dir, exist_ok=True)
print(f"Starting direct download to: {out_dir}")

# WARNING: [0:1] slices the array to only download the VERY FIRST tile for testing.
# Once it succeeds, remove the slice [0:1] to loop through all ~98 tiles.
for i, coords in enumerate(coords_array):
    geom = ee.Geometry.Polygon(coords)
    filename = os.path.join(out_dir, f'FL_Training_Tile_{i + 1:03d}.tif')
    
    print(f"[{i+1}/{total_tiles}] Downloading directly to disk. This might take a few minutes...")
    
    try:
        geemap.download_ee_image(
            image=training_stack,
            filename=filename,
            region=geom,
            crs='EPSG:3857', 
            scale=10
        )
        print(f"Success! Saved: {filename}")
    except Exception as e:
        print(f"Failed to download tile {i+1}. Error: {e}")

print("\nDownload script finished!")

Calculating tile intersections via Raster Engine...

SUCCESS! Total Intersecting Tiles Found: 102
Starting direct download to: c:\vscode workspace\ml-mangrove\Scale Adaption\GEE\Florida_Training_Dataset
[1/102] Downloading directly to disk. This might take a few minutes...


  0%|          |0/690 tiles [00:00<?]

Success! Saved: c:\vscode workspace\ml-mangrove\Scale Adaption\GEE\Florida_Training_Dataset\FL_Training_Tile_001.tif
[2/102] Downloading directly to disk. This might take a few minutes...


  0%|          |0/690 tiles [00:00<?]

Success! Saved: c:\vscode workspace\ml-mangrove\Scale Adaption\GEE\Florida_Training_Dataset\FL_Training_Tile_002.tif
[3/102] Downloading directly to disk. This might take a few minutes...


...

Success! Saved: c:\vscode workspace\ml-mangrove\Scale Adaption\GEE\Florida_Training_Dataset\FL_Training_Tile_100.tif
[101/102] Downloading directly to disk. This might take a few minutes...


  0%|          |0/690 tiles [00:00<?]

Success! Saved: c:\vscode workspace\ml-mangrove\Scale Adaption\GEE\Florida_Training_Dataset\FL_Training_Tile_101.tif
[102/102] Downloading directly to disk. This might take a few minutes...


  0%|          |0/690 tiles [00:00<?]

Success! Saved: c:\vscode workspace\ml-mangrove\Scale Adaption\GEE\Florida_Training_Dataset\FL_Training_Tile_102.tif

Download script finished!
